<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/STA1403_Chapter_12B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_12B:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY _K_ IND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Clustering**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* 12.1 : Introduction
* 12.2 : _K_ -Means Clustering
* **12.3 : Hierarchical Clustering**
* **12.4 : Advanced**

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 KEY

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 key: {e}")
    print("Please set your STA1403 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 26
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your STA1403 key is correctly installed.

## Load Data Sets

Run the next cell to load the data sets for this lesson.

In [ ]:
# @title Load Data Sets
import pandas as pd

# Load dataset
df_protein = pd.read_csv("https://biologicslab.co/STA1403/data/Protein.txt",
                         sep = ' ', na_values=[" ", "NA", "null", ""])

# Load dataset
df_gapminder = pd.read_csv("https://biologicslab.co/STA1403/data/gapminder_2007.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])



print("Data sets loaded sucessfully. ✅")

# **Chapter 12 : Clustering**

## **12.3 Hierarchical Clustering**

There are two potential problems with the $K$-means clustering algorithm. First, it is a flat clustering method. After observations are assigned to their clusters, they are all considered to be similar within the same cluster. That is, the observations are not further separated based on dissimilarity within a cluster. Secondly, we need to specify the number of clusters $K$ _a priori_. Finding the appropriate number of clusters is not trivial, and the selected number has a substantial impact on the results.

An alternative approach that avoids these issues is **hierarchical clustering**. The result of this method is a **dendrogram** (a tree). The $root$ of the dendrogram is its highest level and contains all $n$ observations. The $leaves$ of the tree are its lowest level and are each a unique observation.

--------------------

There are two general algorithms for hierarchical clustering [11]:
1. **Agglomerative** (bottom-up): We start at the bottom of the tree, where every observation is a cluster (i.e., there are $n$ clusters). Then we merge two of the clusters with the smallest degree of dissimilarity (i.e., the two most similar clusters). Now we have $n - 1$ clusters. We continue merging clusters until we have only one cluster (the root) that includes all observations.
2. **Divisive** (top-down): We start at the top of the tree, where all observations are grouped in a single cluster. Then we divide the cluster into two new clusters that are most dissimilar. Now we have two clusters. We continue splitting existing clusters until every observation is its own cluster.

---------------

Of the above two strategies, agglomerative algorithm is more common. Both algorithms, however, require a measure of dissimilarity between two clusters. In other words, we need to specify a distance measure for two clusters analogous to the distance measure we defined for two observations. For every pair of observations, where one is from cluster $i$, and the other one is from cluster $j$, we can find the squared Euclidean distances $d_{ij}$. Then we can use one of the following methods to calculate the overall distance between two clusters:

--------------
    
•&nbsp;_Single linkage_ clustering uses the minimum $ d_{ij} $ among all possible pairs as the distance between the two clusters. This is the distance between two observations, one from each cluster, that are closest to each other.

•&nbsp;_Complete linkage_ clustering uses the maximum $ d_{ij} $ as the distance between the two clusters. This is the distance between two observations, one from each cluster, that are furthest apart.

•&nbsp;_Average linkage_ clustering uses the average $ d_{ij} $ over all possible pairs as the distance between the two clusters.

•&nbsp;_Centroid linkage_ clustering finds the centroids of the two clusters and uses the distance between the centroids as the distance between the two clusters.
    
As an illustration of these methods, consider Fig. 12.5, which shows two clusters shown as circles and squares. The solid line shows the single linkage distance between the two clusters, the dashed line shows the complete linkage distance, and the dotted line shows the centroid linkage distance. Note that the dotted line connects the centers (as opposed to observations) of the two clusters. There are of course other ways for defining the distance between two clusters. However, the above measures are the most commonly used.

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12A_image02A.png)

>**Fig. 12.5** Illustrating the difference between the single linkage method, the complete linkage method, and the centroid linkage method to determine the distance $ d_{ij} $ between the two clusters shown as circles and squares

Example 1 shows how to perform complete linkage clustering to create a dendrogram of countries based on their protein consumption.

Using the results from hierarchical clustering, we can group observations into K clusters by cutting the dendrogram through K branches. For example, cutting the dendrogram in Fig. 12.6 at the dashed line would create three clusters. The first cluster includes three of Balkan countries. The second cluster includes Scandinavian countries and mostly Western countries, and the last cluster mostly consists of countries from Eastern Europe and the Mediterranean.

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12A_image03A.png)

>**Fig. 12.6 The dendrogram resulting from complete linkage clustering of the 25 countries based on their protein consumption. The dashed line shows where to cut the dendrogram to create three clusters**


## Dendogram Analysis with Python

The code in the cell shows how to recreate the dendogram shown in Fig. 12.6 using Python.

In [ ]:
# @title Dendogram Analysis with Python

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = df_protein       # DataFrame
feature_A     = "Country"        # Column for analysis
# ──────────────────────────────────────────────────────────────────────────────

# Feature selection (excluding 'Country')
features = df.drop(columns=[feature_A]).values

# Complete Linkage Clustering with Squared-Euclidean distance
Z = linkage(features, method='complete', metric='sqeuclidean')

# Visualization
plt.figure(figsize=(8, 6))
plt.xlabel(f"{feature_A}", fontsize=12)
plt.ylabel("Height (Distance)", fontsize=12)

# Adding the threshold line mentioned in the prompt
plt.axhline(y=750, color='black', linestyle='--', label='Cluster Threshold')

dendrogram(
    Z,
    labels=df_protein['Country'].tolist(),
    orientation='top',
    leaf_font_size=8
)

plt.tight_layout()
plt.show()

If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12B_image01A.png)

The output shows the dendrogram created by Python that is similar to the one shown in Fig. 12.6. The clusters seemed to be defined by geographic location: Balkan countries (Romania, Bulgaria, and Yugoslavia), Scandinavian countries (Finland, Norway, Denmark, and Sweden), Western European countries (UK, Belgium, France, Austria, Ireland, Switzerland, Netherlands, and West Germany), Eastern European countries (East Germany, Hungary, Czechoslovakia, Poland, Albania, USSR) and the Mediterranean countries (Portugal, Spain, Greece, Italy). (This data set was collected in 1973. Since then, some of these countries have changed or no longer exist.)

Using the results from hierarchical clustering, we can group observations into $K$ clusters by cutting the dendrogram through $K$ branches. For example, cutting the dendrogram in Fig. 12.6 at the dashed line would create three clusters. The first cluster includes three of Balkan countries. The second cluster includes Scandinavian countries and mostly Western countries, and the last cluster mostly consists of countries from Eastern Europe and the Mediterranean.

For the examples and exercises in this lesson, we will be using the `gapminder` dataset.

### Dataset: `gapminder_2007.csv`

The `gapminder_2007.csv` dataset is a snapshot of global development indicators for **142 countries**
in the year **2007**. It is derived from the [Gapminder](https://www.gapminder.org/) project, which
tracks health and economic data across nations over time.

#### **Columns**

| Column | Type | Description |
|---|---|---|
| `country` | string | Country name |
| `continent` | string | Continent (`Africa`, `Americas`, `Asia`, `Europe`, `Oceania`) |
| `lifeExp` | float | Life expectancy at birth (years) |
| `pop` | integer | Total population |
| `gdpPercap` | float | GDP per capita (USD, inflation-adjusted) |

#### **Continent Breakdown**

| Continent | Countries |
|---|---|
| Africa | 52 |
| Asia | 33 |
| Europe | 30 |
| Americas | 25 |
| Oceania | 2 |

### Notes for Clustering Analysis

- **Standardize before clustering.** The three numeric features (`lifeExp`, `pop`, `gdpPercap`) are on
  very different scales. Apply `StandardScaler` before computing the linkage matrix to prevent
  population size from dominating the distance calculation.
- **Filter by continent** to reduce the number of labels in the dendrogram and reveal finer
  within-region structure.
- **Expected cluster patterns:** High-income countries (Western Europe, North America, Oceania)
  tend to cluster together on high GDP and life expectancy, while many Sub-Saharan African nations
  form a separate group characterized by lower values on both measures.

The `gapminder_2007` dataset has been stored in a DataFrame called `df_gapminder`. Run the next cell to see the first 5 records in this DataFrame.

In [ ]:
# @title gapminder dataset

df_gapminder.head()

## Example 1: Dendogram Analysis

The code in the cell below show how to generate a dendogram of the `gapminder` dataset for the continent of `Europe`.

In [ ]:
# @title Example 1: Dendogram Analysis

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A  = "country"
feature_B  = "continent"
selected_continent = "Europe"
# ──────────────────────────────────────────────────────────────────────────────

# Filter to selected continent
subset = df_gapminder[df_gapminder[feature_B] == selected_continent].reset_index(drop=True)

# Feature selection (excluding country and continent)
features = subset.drop(columns=[feature_A, feature_B]).values

# Complete Linkage Clustering with Squared-Euclidean distance
Z = linkage(features, method='complete', metric='sqeuclidean')

# Visualization
plt.figure(figsize=(10, 6))
plt.xlabel(f"{feature_A}", fontsize=12)
plt.ylabel("Height (Distance)", fontsize=12)
plt.title(f"Hierarchical Clustering — {selected_continent}", fontsize=13)

plt.axhline(y=750, color='black', linestyle='--', label='Cluster Threshold')

dendrogram(
    Z,
    labels=subset[feature_A].tolist(),
    orientation='top',
    leaf_font_size=10
)

plt.tight_layout()
plt.show()

If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12B_image02A.png)

Since the variables were _not_ standardized, the dendogram is somewhat difficult to interpret.

## **Exercise 1: Dendogram Analysis**

In the cell below, write the code to generate a dendogram of the `gapminder` dataset for the continent of `Americas`.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below.
2. Change the configuation to read as follows:
```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A  = "country"
feature_B  = "continent"
selected_continent = "Americas"   
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 1: Dendogram Analysis




If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12B_image03A.png)

Again, it is difficult to interpret the dendogram since the variables have not been standardized.

## **12.4 Advanced**
In this section, we discuss standardization of variables before clustering.

### **_12.4.1 Standardizing Variables Before Clustering_**

For distance-based clustering methods discussed in this chapter, it is common to standardize variable prior to clustering so that all variables contribute equally to the overall distance measure and have the same influence on the results. To this end, we divide each variable by its standard deviation as discussed in Sect. 2.6. This way, all variables have standard deviations of 1, so they become comparable.

Example 2 repeats the analysis performed in Example 1, but standardized the dataset before plotting the dendogram.

## Example 2: Dendogram Analysis with Standardization

To standardize the datset before plotting the dendogram, the code in the cell below adds this code snippet:
```Python
# Normalize features (zero mean, unit variance)
scaler = StandardScaler()
features = scaler.fit_transform(features_raw)
```


In [ ]:
# @title Example 2: Dendogram Analysis with Standarization
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A  = "country"
feature_B  = "continent"
selected_continent = "Europe"
# ──────────────────────────────────────────────────────────────────────────────

# Filter to selected continent
subset = df_gapminder[df_gapminder[feature_B] == selected_continent].reset_index(drop=True)

# Feature selection (excluding country and continent)
features_raw = subset.drop(columns=[feature_A, feature_B]).values

# Normalize features (zero mean, unit variance)
scaler = StandardScaler()
features = scaler.fit_transform(features_raw)

# Complete Linkage Clustering with Squared-Euclidean distance
Z = linkage(features, method='complete', metric='sqeuclidean')

# Visualization
plt.figure(figsize=(10, 6))
plt.xlabel(f"{feature_A}", fontsize=12)
plt.ylabel("Height (Distance)", fontsize=12)
plt.title(f"Hierarchical Clustering — {selected_continent}", fontsize=13)
plt.axhline(y=750, color='black', linestyle='--', label='Cluster Threshold')
dendrogram(
    Z,
    labels=subset[feature_A].tolist(),
    orientation='top',
    leaf_font_size=10
)
plt.tight_layout()
plt.show()

If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12B_image05A.png)

In general, not all variables have the same degree of importance in grouping observations into clusters. In some situations, giving all variables the same influence by standardizing them could lead to obscuring the patterns and misleading the clustering methods. Possible pitfalls of standardizing variables before clustering is illustrated in Fig. 14.5 of “The Elements of Statistical Learning” by Hastie et al. $ ^{[11]} $ .

## **Exercise 2: Dendogram Analysis with Standardization**

Repeat **Exercise 1** but this time, standardize the variables.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A  = "country"
feature_B  = "continent"
selected_continent = "Americas"   
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 2: Dendogram Analysis with Standarization




If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12B_image06A.png)

It should be noted that some of the "leaf" names at the bottom of the dendogram are overlapping due to the relatively small size (i.e. width) of the graphic so it can be displayed in Google Colab.  

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()